# Lab 2 — Ứng dụng Pre-trained Models trên CIFAR-10

> **Môn học:** Deep Learning — **Chủ đề:** Transfer Learning & Fine-tuning bằng PyTorch  
> **Tập dữ liệu:** CIFAR-10 (60.000 ảnh màu 32×32, 10 lớp)

Notebook này điều phối toàn bộ quy trình phân loại CIFAR-10 bằng các mô hình pretrained VGG16 và ResNet-18. Trên Colab, nguồn code được lấy từ GitHub; dữ liệu và kết quả cần lưu lâu dài được đặt trên Google Drive.

## Bản tóm tắt nội dung

| Mục | Nội dung | Phạm vi / trạng thái |
|---|---|---|
| **0. Setup & Run Configuration** | Nguồn code, đường dẫn Drive/runtime, chọn `RUN_ID` | Phase 8: cấu hình và kiểm tra, không huấn luyện |
| **1. CIFAR-10 Data** | Kiểm tra dữ liệu và tạo train/validation/test loaders | Chỉ chạy sau khi cấu hình Drive/xác thực dataset |
| **2. Pretrained Architectures** | Khởi tạo và khám phá kiến trúc | Pipeline hiện có; chưa liên hoàn với sáu run |
| **3. CIFAR-10 Model Adaptation** | Classifier, đóng gói và BatchNorm | Pipeline hiện có; chưa liên hoàn với `RUN_ID` |
| **4. Training & Validation** | Huấn luyện, theo dõi và lưu checkpoint | Ngoài phạm vi Phase 8; không chạy ở bước cấu hình |
| **5. Model Comparison** | So sánh lịch sử các run | Thực hiện sau khi có kết quả |
| **6. ResNet-18 Fine-tuning** | Fine-tune từ checkpoint baseline | Thực hiện sau khi baseline hoàn tất |
| **7. Final Test Evaluation** | Đánh giá cuối trên test set | Chỉ sau khi chốt cấu hình; không dùng để chọn hyperparameter |

## Quy ước dữ liệu

- CIFAR-10 được chia thành **45.000 train**, **5.000 validation** và **10.000 test**.
- Train dùng để cập nhật weights; validation dùng để chọn checkpoint/cấu hình.
- Test chỉ dùng để đánh giá cuối sau khi mô hình/run đã được chốt.

> **Giới hạn Phase 8:** Phạm vi hiện tại là chuẩn hóa setup và cấu hình. Các cell experiment phía dưới cần giữ nguyên cho VGG16, ResNet-18 và fine-tune; chúng chưa được điều khiển bởi `RUN_ID` và không chạy tuần tự qua các cell experiment trong Phase 8.



---
## 0. Setup & Run Configuration

Cell setup phân biệt code checkout, nguồn CIFAR-10 trên Drive, thư mục dữ liệu runtime, artifacts và TensorBoard logs. Cell cấu hình tiếp theo chỉ xác thực và hiển thị một `RUN_ID` cùng các tham số tương ứng.

**Trong Phase 8:** không chạy huấn luyện, resume, inference reuse hay đánh giá test. Trên Colab, chỉ thực thi setup sau khi đã đưa phiên bản notebook/code lên GitHub và chuẩn bị cấu trúc Drive theo hướng dẫn.



In [ ]:
from pathlib import Path
import subprocess
import sys

GITHUB_REPO_URL = "https://github.com/toan58936/lab2.git"
GITHUB_BRANCH = "main"
GITHUB_COMMIT = None  # Set a commit hash to pin the exact source revision.

try:
    import google.colab
    IS_COLAB = True
except ModuleNotFoundError:
    IS_COLAB = False

if IS_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    CODE_ROOT = Path("/content/lab2")
    DRIVE_ROOT = Path("/content/drive/MyDrive/lab2")
    DRIVE_DATA_ROOT = DRIVE_ROOT / "data"
    DRIVE_ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
    DRIVE_RUNS_ROOT = DRIVE_ROOT / "runs"
    DATA_DIR = Path("/content/cifar10_data")

    if not CODE_ROOT.exists():
        subprocess.run(
            ["git", "clone", "--branch", GITHUB_BRANCH, "--single-branch", GITHUB_REPO_URL, str(CODE_ROOT)],
            check=True,
        )
    elif not (CODE_ROOT / ".git").exists():
        raise FileNotFoundError(
            f"{CODE_ROOT} exists but is not a Git checkout. Rename it before cloning."
        )

    if GITHUB_COMMIT is not None:
        subprocess.run(["git", "-C", str(CODE_ROOT), "checkout", GITHUB_COMMIT], check=True)

    CODE_REVISION = subprocess.run(
        ["git", "-C", str(CODE_ROOT), "rev-parse", "HEAD"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    ARTIFACT_ROOT = DRIVE_ARTIFACT_ROOT
    RUNS_ROOT = DRIVE_RUNS_ROOT
else:
    CODE_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DRIVE_ROOT = None
    DRIVE_DATA_ROOT = None
    DRIVE_ARTIFACT_ROOT = None
    DRIVE_RUNS_ROOT = None
    DATA_DIR = CODE_ROOT / "data"
    ARTIFACT_ROOT = CODE_ROOT / "artifacts"
    RUNS_ROOT = CODE_ROOT / "runs"
    CODE_REVISION = "local working tree"

CODE_ROOT = CODE_ROOT.resolve()
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

print(f"CODE_ROOT      : {CODE_ROOT}")
print(f"CODE_REVISION  : {CODE_REVISION}")
print(f"DRIVE_ROOT     : {DRIVE_ROOT}")
print(f"DRIVE_DATA_ROOT: {DRIVE_DATA_ROOT}")
print(f"DATA_DIR       : {DATA_DIR}")
print(f"ARTIFACT_ROOT  : {ARTIFACT_ROOT}")
print(f"RUNS_ROOT      : {RUNS_ROOT}")
print(f"Colab          : {IS_COLAB}")


In [ ]:
import torch
from torch import nn
import torchvision.models as models
import matplotlib.pyplot as plt
import pandas as pd
from timeit import default_timer as timer

try:
    from torch.utils.tensorboard import SummaryWriter
except ImportError:
    raise ImportError('Cài tensorboard: pip install tensorboard')

from going_modular import data_setup, engine, evaluate, model_builder, utils

print(f'PyTorch   : {torch.__version__}')
print(f'CUDA      : {torch.cuda.is_available()}')


In [ ]:
# ============================================================
# PHASE 8: RESOLVE CONFIGURATION ONLY; TRAINING IS OUT OF SCOPE
# ============================================================
SEED = 42
VAL_SIZE = 5_000
NUM_WORKERS = 0
RUN_MODE = "train"  # "train" or "reuse" for the later execution phase.
RESUME = False       # Only applies to RUN_MODE == "train".

ALLOWED_RUN_IDS = (
    "v0_vgg16",
    "v1_resnet18",
    "v1_resnet18_finetune",
    "v2_resnet18_lr1e-2",
    "v2_resnet18_lr1e-4",
    "v2_resnet18_bs64",
)
RUN_ID = "v0_vgg16"

if RUN_ID not in ALLOWED_RUN_IDS:
    raise ValueError(f"Unknown RUN_ID {RUN_ID!r}; choose one of {ALLOWED_RUN_IDS}.")
if RUN_MODE not in {"train", "reuse"}:
    raise ValueError("RUN_MODE must be 'train' or 'reuse'.")
if RESUME and RUN_MODE != "train":
    raise ValueError("RESUME=True is only valid when RUN_MODE='train'.")

RUN_SPECS = {
    "v0_vgg16": {
        "arch": "vgg16", "weights": "IMAGENET1K_V1", "freeze_backbone": True,
        "unfreeze_last_n": 0, "epochs": 5, "batch_size": 32, "lr": 1e-3,
        "optimizer": "adam", "param_groups": None, "dependency_run_id": None,
    },
    "v1_resnet18": {
        "arch": "resnet18", "weights": "IMAGENET1K_V1", "freeze_backbone": True,
        "unfreeze_last_n": 0, "epochs": 5, "batch_size": 32, "lr": 1e-3,
        "optimizer": "adam", "param_groups": None, "dependency_run_id": None,
    },
    "v1_resnet18_finetune": {
        "arch": "resnet18", "weights": None, "freeze_backbone": True,
        "unfreeze_last_n": 1, "epochs": 3, "batch_size": 32, "lr": 1e-4,
        "optimizer": "adam", "param_groups": {"head": 1e-3, "backbone": 1e-4},
        "dependency_run_id": "v1_resnet18",
        "source_checkpoint": "v1_resnet18/best_model.pth",
    },
    "v2_resnet18_lr1e-2": {
        "arch": "resnet18", "weights": "IMAGENET1K_V1", "freeze_backbone": True,
        "unfreeze_last_n": 0, "epochs": 5, "batch_size": 32, "lr": 1e-2,
        "optimizer": "adam", "param_groups": None, "dependency_run_id": None,
    },
    "v2_resnet18_lr1e-4": {
        "arch": "resnet18", "weights": "IMAGENET1K_V1", "freeze_backbone": True,
        "unfreeze_last_n": 0, "epochs": 5, "batch_size": 32, "lr": 1e-4,
        "optimizer": "adam", "param_groups": None, "dependency_run_id": None,
    },
    "v2_resnet18_bs64": {
        "arch": "resnet18", "weights": "IMAGENET1K_V1", "freeze_backbone": True,
        "unfreeze_last_n": 0, "epochs": 5, "batch_size": 64, "lr": 1e-3,
        "optimizer": "adam", "param_groups": None, "dependency_run_id": None,
    },
}

spec = RUN_SPECS[RUN_ID]
RUN_ARTIFACT_DIR = ARTIFACT_ROOT / RUN_ID
RUN_PATHS = {
    "config": RUN_ARTIFACT_DIR / "config.json",
    "history": RUN_ARTIFACT_DIR / "history.json",
    "best_model": RUN_ARTIFACT_DIR / "best_model.pth",
    "last_checkpoint": RUN_ARTIFACT_DIR / "last_checkpoint.pth",
    "test_metrics": RUN_ARTIFACT_DIR / "test_metrics.json",
}

if spec["dependency_run_id"] is not None:
    dependency_checkpoint = (
        ARTIFACT_ROOT / spec["dependency_run_id"] / "best_model.pth"
    )
    if RUN_MODE == "train" and not dependency_checkpoint.is_file():
        raise FileNotFoundError(
            f"{RUN_ID} requires the completed baseline checkpoint: {dependency_checkpoint}"
        )

BATCH_SIZE = spec["batch_size"]
EPOCHS = spec["epochs"]
LR = spec["lr"]

print(f"RUN_ID          : {RUN_ID}")
print(f"RUN_MODE        : {RUN_MODE}")
print(f"RESUME          : {RESUME}")
print(f"Resolved spec   : {spec}")
print(f"Data source     : {DRIVE_DATA_ROOT if IS_COLAB else DATA_DIR}")
print(f"DataLoader path : {DATA_DIR}")
print(f"Run artifacts   : {RUN_ARTIFACT_DIR}")
print(f"Run paths       : {RUN_PATHS}")
print(f"Seed={SEED} | Val_size={VAL_SIZE} | Num_workers={NUM_WORKERS}")


---
## 1. CIFAR-10 Data

Phần này xác thực dữ liệu CIFAR-10 để chuẩn bị trên Drive, có thể stage sang thư mục runtime của Colab, rồi tạo ba DataLoader. Dataset phải có các batch CIFAR-10 chuẩn và `batches.meta`; DataLoader được gọi với `download=False` để không tự tải dữ liệu ngoài cấu hình.

Train và validation được tách theo seed từ tập train gốc; test được giữ riêng cho đánh giá cuối. Việc đọc batch thực thi sau khi người dùng cấu hình Drive, không thuộc bước kiểm tra code của các bậc.



In [ ]:
import shutil
import tarfile
import tempfile
from pathlib import PurePosixPath

_CIFAR_REQUIRED_FILES = {
    "data_batch_1", "data_batch_2", "data_batch_3", "data_batch_4",
    "data_batch_5", "test_batch", "batches.meta",
}


def _valid_cifar10_directory(path: Path) -> bool:
    return path.is_dir() and _CIFAR_REQUIRED_FILES.issubset(
        {item.name for item in path.iterdir() if item.is_file()}
    )


def _stage_cifar10_from_drive() -> None:
    destination = DATA_DIR / "cifar-10-batches-py"
    if _valid_cifar10_directory(destination):
        return
    if destination.exists():
        raise FileExistsError(
            f"Incomplete CIFAR-10 directory at {destination}; it will not be overwritten."
        )

    source_directory = DRIVE_DATA_ROOT / "cifar-10-batches-py"
    if not _valid_cifar10_directory(source_directory) and _valid_cifar10_directory(DRIVE_DATA_ROOT):
        source_directory = DRIVE_DATA_ROOT

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    if _valid_cifar10_directory(source_directory):
        shutil.copytree(source_directory, destination)
    else:
        archive_path = DRIVE_DATA_ROOT / "cifar-10-python.tar.gz"
        if not archive_path.is_file():
            raise FileNotFoundError(
                "CIFAR-10 was not found on Drive. Upload either "
                f"{source_directory} or {archive_path}."
            )

        with tempfile.TemporaryDirectory(prefix="cifar10_stage_") as temporary_directory:
            with tarfile.open(archive_path, "r:gz") as archive:
                members = archive.getmembers()
                for member in members:
                    member_path = PurePosixPath(member.name)
                    if (
                        member_path.is_absolute()
                        or ".." in member_path.parts
                        or not member_path.parts
                        or member_path.parts[0] != "cifar-10-batches-py"
                        or not (member.isfile() or member.isdir())
                    ):
                        raise ValueError(
                            f"Unexpected or unsafe member in CIFAR-10 archive: {member.name}"
                        )
                archive.extractall(temporary_directory, members=members)

            extracted_directory = Path(temporary_directory) / "cifar-10-batches-py"
            if not _valid_cifar10_directory(extracted_directory):
                raise ValueError(
                    f"The archive does not contain a valid CIFAR-10 dataset: {archive_path}"
                )
            shutil.copytree(extracted_directory, destination)

    if not _valid_cifar10_directory(destination):
        raise ValueError(f"Staged CIFAR-10 dataset is incomplete: {destination}")


if IS_COLAB:
    _stage_cifar10_from_drive()
elif not _valid_cifar10_directory(DATA_DIR / "cifar-10-batches-py"):
    raise FileNotFoundError(
        f"CIFAR-10 files were not found under {DATA_DIR}; prepare the dataset before loading."
    )

train_dl, val_dl, test_dl, CLASS_NAMES = data_setup.create_train_val_test_dataloaders(
    batch_size=BATCH_SIZE,
    data_dir=str(DATA_DIR),
    transform_model="vgg16",
    val_size=VAL_SIZE,
    seed=SEED,
    num_workers=NUM_WORKERS,
    download=False,
)

NUM_CLASSES = len(CLASS_NAMES)
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"Train batches : {len(train_dl):>5}  (~{len(train_dl) * BATCH_SIZE:>6} samples)")
print(f"Val batches   : {len(val_dl):>5}  (~{len(val_dl) * BATCH_SIZE:>6} samples)")
print(f"Test batches  : {len(test_dl):>5}  (~{len(test_dl) * BATCH_SIZE:>6} samples)")


In [ ]:
# -- Kiểm tra shape & dtype của một batch --
images, labels = next(iter(train_dl))
print(f'Batch images : {images.shape}   # [B, C, H, W]')
print(f'Batch labels : {labels.shape}')
print(f'Dtype        : {images.dtype}')
print(f'Pixel range (ImageNet-normalized): [{images.min():.3f}, {images.max():.3f}]')

In [ ]:
# -- De-normalize để hiển thị ảnh --
# Tensor đã normalize theo ImageNet, cần đảo ngược trước khi plt.imshow()
from torchvision.models import VGG16_Weights
WEIGHT_TRANSFORM = VGG16_Weights.IMAGENET1K_V1.transforms()
IMAGENET_MEAN = torch.tensor(WEIGHT_TRANSFORM.mean).view(3, 1, 1)
IMAGENET_STD  = torch.tensor(WEIGHT_TRANSFORM.std).view(3, 1, 1)

def denormalize(t):
    """[C,H,W] normalized → [C,H,W] in [0,1]"""
    return (t * IMAGENET_STD + IMAGENET_MEAN).clamp(0.0, 1.0)

torch.manual_seed(SEED)
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax, img, lbl in zip(axes.flat, images[:16], labels[:16]):
    ax.imshow(denormalize(img).permute(1, 2, 0))
    ax.set_title(CLASS_NAMES[lbl.item()], fontsize=9)
    ax.axis('off')
fig.suptitle('16 ảnh train (de-normalized, center crop 224×224)', fontsize=13)
plt.tight_layout()
plt.show()

---
## 2. Pretrained Architectures

Phần này khởi tạo và khám phá kiến trúc VGG16, ResNet-18 kèm ImageNet weights, sau đó xem cấu trúc và số lượng tham số. Ở chế độ train, việc tải pretrained weights cần mạng và thuộc giai đoạn thực nghiệm.

Trong Phase 8, không chạy các cell khởi tạo model nếu chưa tải weights; chỉ hoàn thiện cấu hình notebook.



In [ ]:
utils.set_seed(SEED)

# Trong reuse, kh?i t?o ki?n tr?c kh?ng t?i weights ImageNet; checkpoint ??y ?? s? ???c n?p ? Section 4.
LOAD_PRETRAINED = RUN_MODE == "train"
cfg_v0 = ModelConfig(arch='vgg16', num_classes=NUM_CLASSES, weights='IMAGENET1K_V1' if LOAD_PRETRAINED else None, freeze_backbone=True)
cfg_v1 = ModelConfig(arch='resnet18', num_classes=NUM_CLASSES, weights='IMAGENET1K_V1' if LOAD_PRETRAINED else None, freeze_backbone=True)
model_v0 = build_model(cfg_v0)
model_v1 = build_model(cfg_v1)

# -- B?ng so s?nh params --
rows = []
for name, m in [('v0 ? VGG16', model_v0), ('v1 ? ResNet-18', model_v1)]:
    stats = count_params(m)
    total = stats['total']
    trainable = stats['trainable']
    rows.append({'Model': name,
                 'Total params': f'{total:,}',
                 'Trainable': f'{trainable:,}',
                 'Frozen': f'{total - trainable:,}',
                 'Trainable %': f'{100*trainable/total:.1f}%'})

pd.DataFrame(rows).set_index('Model')


In [ ]:
# -- Xem kiến trúc V0 (VGG16) --
print('=' * 60)
print('V0 — VGG16 pretrained ImageNet  |  freeze_backbone=True')
print('=' * 60)
print(describe_model(model_v0))

In [ ]:
# -- Xem kiến trúc V1 (ResNet-18) --
print('=' * 60)
print('V1 — ResNet-18 pretrained ImageNet  |  freeze_backbone=True')
print('=' * 60)
print(describe_model(model_v1))

---
## 3. CIFAR-10 Model Adaptation

Phần này minh họa thay lớp classifier để tạo đầu ra 10 lớp, đồng봉 backbone và giữ BatchNorm ở trạng thái phù hợp khi train.

| Chiến lược | Nghĩa |
|---|---|
| **Feature extraction** | Đóng băng backbone, chỉ cập nhật classifier mới |
| **Fine-tuning** | Mở khóa một hoặc nhiều block cuối và cập nhật cùng classifier |
| **Full training** | Cập nhật toàn bộ mạng; không phải cấu hình của các run Phase 8 |

Các cell model hiện nay còn theo pipeline VGG16/ResNet-18 cứng; cần được nối với `RUN_SPECS[RUN_ID]` trước khi dùng cho sáu run.



In [ ]:
# -- Verify output shape: phải là [B, 10] cho CIFAR-10 --
dummy = torch.randn(2, 3, 224, 224)
with torch.inference_mode():
    out_v0 = model_v0(dummy)
    out_v1 = model_v1(dummy)

assert out_v0.shape == (2, NUM_CLASSES), f'v0 shape sai: {out_v0.shape}'
assert out_v1.shape == (2, NUM_CLASSES), f'v1 shape sai: {out_v1.shape}'
print(f'v0 (VGG16)    output: {out_v0.shape}  ✓')
print(f'v1 (ResNet18) output: {out_v1.shape}  ✓')

In [ ]:
# -- Xem lớp classifier đã được thay --
print('V0 classifier (lớp cuối VGG16):')
print(model_v0.classifier[-1])   # Linear(4096 -> 10)
print()
print('V1 classifier (fc layer ResNet-18):')
print(model_v1.fc)          # Linear(512 -> 10)

---
## 4. Training & Validation — ngoài phạm vi cấu hình Phase 8

Các cell trong mục này chưa lệnh `engine.train` cho hai luồng cố định: VGG16 và ResNet-18. Chúng chưa lấy toàn bộ tham số, artifact path, dependency hay resume state từ `RUN_SPECS[RUN_ID]`.

**Không chạy các cell 4.1–4.2 trong Phase 8.** Việc tách hợp train/resume/reuse theo từng `RUN_ID` sẽ thuộc phase thực nghiệm sau.



In [ ]:
def log_run_config(writer, config: dict):
    """Ghi run config vào TensorBoard text tab."""
    lines = [f'`{k}` = `{v}`' for k, v in config.items()]
    writer.add_text('run_config', '  \n'.join(lines), global_step=0)

loss_fn = nn.CrossEntropyLoss()

---
### 4.1. VGG16 — pipeline train hiện có (chưa tách hợp `RUN_ID`)

Cell experiment cố định cho VGG16. Không chạy trong Phase 8 cấu hình.



In [ ]:
ARTIFACT_V0 = ARTIFACT_ROOT / 'v0_vgg16'
CHECKPOINT_V0 = str(ARTIFACT_V0 / 'best_model.pth')
CONFIG_V0 = {
    'model': 'VGG16', 'weights': 'ImageNet1K_V1', 'num_classes': NUM_CLASSES,
    'freeze_backbone': True, 'batch_size': BATCH_SIZE,
    'lr': LR, 'epochs': EPOCHS, 'seed': SEED,
}
print(f"VGG16 artifacts: {ARTIFACT_V0}")


In [ ]:
print('-' * 55)
print(f'VGG16 | mode={RUN_MODE} | freeze_backbone=True')
print('-' * 55)

if RUN_MODE == 'train':
    utils.set_seed(SEED)
    optimizer_v0 = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model_v0.parameters()), lr=LR
    )
    writer_v0 = SummaryWriter(log_dir=str(RUNS_ROOT / 'v0_vgg16'))
    log_run_config(writer_v0, CONFIG_V0)

    start_v0 = timer()
    try:
        history_v0 = engine.train(
            model=model_v0,
            train_dataloader=train_dl,
            val_dataloader=val_dl,
            optimizer=optimizer_v0,
            loss_fn=loss_fn,
            epochs=EPOCHS,
            device=DEVICE,
            writer=writer_v0,
            save_path=CHECKPOINT_V0,
            checkpoint_dir=str(ARTIFACT_V0),
            resume=False,
            expected_phase_config=CONFIG_V0,
            start_epoch=0,
            verbose=False,
        )
    finally:
        writer_v0.close()
    time_v0 = timer() - start_v0
    CONFIG_V0['train_seconds'] = time_v0
    utils.save_config(str(ARTIFACT_V0), CONFIG_V0)
    utils.save_history(str(ARTIFACT_V0), history_v0)
else:
    history_v0, saved_config_v0 = load_experiment(str(ARTIFACT_V0))
    utils.load_model(model_v0, str(ARTIFACT_V0 / 'best_model.pth'), device=DEVICE)
    time_v0 = float(saved_config_v0.get('train_seconds', 0.0))
    print(f"Loaded saved config: {saved_config_v0}")

best_v0 = max(history_v0['val_acc'])
print(f'VGG16 ready | train time: {time_v0:.1f}s | best val_acc: {best_v0:.4f}')


---
### 4.2. ResNet-18 — pipeline train hiện có (chưa tách hợp `RUN_ID`)

Cell experiment cố định cho ResNet-18. Không chạy trong Phase 8 cấu hình. Khi tách hợp, callback train mode cần giữ BatchNorm của backbone ở eval mode.



In [ ]:
ARTIFACT_V1 = ARTIFACT_ROOT / 'v1_resnet18'
CHECKPOINT_V1 = str(ARTIFACT_V1 / 'best_model.pth')
CONFIG_V1 = {
    'model': 'ResNet-18', 'weights': 'ImageNet1K_V1', 'num_classes': NUM_CLASSES,
    'freeze_backbone': True, 'batch_size': BATCH_SIZE,
    'lr': LR, 'epochs': EPOCHS, 'seed': SEED,
}
print(f"ResNet-18 artifacts: {ARTIFACT_V1}")


In [ ]:
print('-' * 55)
print(f'ResNet-18 | mode={RUN_MODE} | freeze_backbone=True')
print('-' * 55)

if RUN_MODE == 'train':
    utils.set_seed(SEED)
    optimizer_v1 = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model_v1.parameters()), lr=LR
    )
    writer_v1 = SummaryWriter(log_dir=str(RUNS_ROOT / 'v1_resnet18'))
    log_run_config(writer_v1, CONFIG_V1)

    start_v1 = timer()
    try:
        history_v1 = engine.train(
            model=model_v1,
            train_dataloader=train_dl,
            val_dataloader=val_dl,
            optimizer=optimizer_v1,
            loss_fn=loss_fn,
            epochs=EPOCHS,
            device=DEVICE,
            writer=writer_v1,
            save_path=CHECKPOINT_V1,
            checkpoint_dir=str(ARTIFACT_V1),
            resume=False,
            expected_phase_config=CONFIG_V1,
            start_epoch=0,
            train_mode_fn=lambda m: set_train_mode(m, ModelConfig(arch='resnet18', freeze_bn_stats=True)),
            verbose=False,
        )
    finally:
        writer_v1.close()
    time_v1 = timer() - start_v1
    CONFIG_V1['train_seconds'] = time_v1
    utils.save_config(str(ARTIFACT_V1), CONFIG_V1)
    utils.save_history(str(ARTIFACT_V1), history_v1)
else:
    history_v1, saved_config_v1 = load_experiment(str(ARTIFACT_V1))
    utils.load_model(model_v1, str(ARTIFACT_V1 / 'best_model.pth'), device=DEVICE)
    time_v1 = float(saved_config_v1.get('train_seconds', 0.0))
    print(f"Loaded saved config: {saved_config_v1}")

best_v1 = max(history_v1['val_acc'])
print(f'ResNet-18 ready | train time: {time_v1:.1f}s | best val_acc: {best_v1:.4f}')


---
## 5. Model Comparison — sau khi có kết quả

Phần này dùng để kiểm so sánh history, loss/accuracy curves và thời gian giữa các run sau khi hoàn thành. Các biểu đồ vẽ ở hiện tại chỉ phản ánh những biến của pipeline cứng; chưa phải kết quả tập hợp sáu `RUN_ID`.

Không chạy phần so sánh như một bước kiểm chứng cấu hình Phase 8.



In [ ]:
# -- Bảng tóm tắt kết quả training --
summary_data = {
    'Model'           : ['v0 — VGG16', 'v1 — ResNet-18'],
    'Best val_acc'    : [f'{best_v0:.4f}', f'{best_v1:.4f}'],
    'Train time (s)'  : [f'{time_v0:.1f}', f'{time_v1:.1f}'],
    'Trainable params': [
        f"{count_params(model_v0)['trainable']:,}",
        f"{count_params(model_v1)['trainable']:,}",
    ],
}
pd.DataFrame(summary_data).set_index('Model')

In [ ]:
# -- Loss & Accuracy curves --
evaluate.plot_history(
    histories={
        'v0 — VGG16'    : history_v0,
        'v1 — ResNet-18': history_v1,
    },
    show=True,
)

---
## 6. ResNet-18 Fine-tuning — phụ thuộc baseline

Run fine-tune cần khởi tạo từ `best_model.pth` của `v1_resnet18`, sau đó mở khóa `layer4` và classifier với optimizer/LR theo cấu hình đã chốt.

Cell hiện tại là pipeline fine-tune cố định, chưa được điều khiển bởi `RUN_ID`, `RUN_SPECS` và `RESUME`. Chỉ thực hiện ở phase huấn luyện sau khi baseline hoàn tất và cell cần được tách hợp.



In [ ]:
# Kh?i t?o ki?n tr?c r?i n?p checkpoint t?t nh?t c?a ResNet-18.
cfg_ft = ModelConfig(arch='resnet18', num_classes=NUM_CLASSES, weights=None, freeze_backbone=True, unfreeze_last_n=1)
model_v1_ft = build_model(cfg_ft)
model_v1_ft = utils.load_model(model_v1_ft, str(CHECKPOINT_V1), device=DEVICE)

stats_ft = count_params(model_v1_ft)
total_ft = stats_ft['total']
trainable_ft = stats_ft['trainable']
print(f'Sau unfreeze layer4+fc:')
print(f'  Total     : {total_ft:,}')
print(f'  Trainable : {trainable_ft:,}  ({100*trainable_ft/total_ft:.1f}%)')
print(f'  Frozen    : {total_ft - trainable_ft:,}')


In [ ]:
LR_FINETUNE = 1e-4
EPOCHS_FT = 3
ARTIFACT_FT = ARTIFACT_ROOT / 'v1_resnet18_finetune'
CHECKPOINT_FT = str(ARTIFACT_FT / 'best_model.pth')
CONFIG_FT = {
    'model': 'ResNet-18 fine-tuned', 'weights': 'ImageNet1K_V1',
    'starting_checkpoint': 'v1_resnet18/best_model.pth',
    'unfrozen_layers': ['layer4', 'fc'], 'num_classes': NUM_CLASSES,
    'batch_size': BATCH_SIZE, 'lr': LR_FINETUNE,
    'epochs': EPOCHS_FT, 'seed': SEED,
}

if RUN_MODE == 'train':
    utils.set_seed(SEED)
    optimizer_ft = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model_v1_ft.parameters()),
        lr=LR_FINETUNE,
    )
    writer_ft = SummaryWriter(log_dir=str(RUNS_ROOT / 'v1_resnet18_finetune'))
    log_run_config(writer_ft, CONFIG_FT)

    print('-' * 55)
    print(f'Fine-tuning ResNet-18 layer4+fc | LR={LR_FINETUNE}')
    print('-' * 55)

    start_ft = timer()
    try:
        history_ft = engine.train(
            model=model_v1_ft,
            train_dataloader=train_dl,
            val_dataloader=val_dl,
            optimizer=optimizer_ft,
            loss_fn=loss_fn,
            epochs=EPOCHS_FT,
            device=DEVICE,
            writer=writer_ft,
            save_path=CHECKPOINT_FT,
            checkpoint_dir=str(ARTIFACT_FT),
            resume=False,
            expected_phase_config=CONFIG_FT,
            start_epoch=0,
            verbose=False,
        )
    finally:
        writer_ft.close()
    time_ft = timer() - start_ft
    CONFIG_FT['train_seconds'] = time_ft
    utils.save_config(str(ARTIFACT_FT), CONFIG_FT)
    utils.save_history(str(ARTIFACT_FT), history_ft)
else:
    history_ft, saved_config_ft = load_experiment(str(ARTIFACT_FT))
    utils.load_model(model_v1_ft, str(ARTIFACT_FT / 'best_model.pth'), device=DEVICE)
    time_ft = float(saved_config_ft.get('train_seconds', 0.0))
    print(f"Loaded saved fine-tuning config: {saved_config_ft}")

best_ft = max(history_ft['val_acc'])
print(f'Fine-tuned model ready | train time: {time_ft:.1f}s | best val_acc: {best_ft:.4f}')


In [ ]:
# -- So sánh: frozen vs fine-tuned --
print('\nSo sánh ResNet-18:')
print(f'  Frozen backbone  : val_acc = {best_v1:.4f}')
print(f'  Fine-tuned layer4: val_acc = {best_ft:.4f}')
print(f'  Cải thiện        : {(best_ft - best_v1)*100:+.2f}%')

evaluate.plot_history(
    histories={
        'v1 frozen'   : history_v1,
        'v1 fine-tune': history_ft,
    },
    show=True,
)

---
## 7. Final Test Evaluation — chỉ chạy sau khi chốt mô hình

> **Chỉ chạy sau khi quá trình huấn luyện và lựa chọn checkpoint bằng validation đã hoàn tất.** Không sử dụng kết quả test để chọn kiến trúc, hyperparameter hay checkpoint.

Phần này tạo metrics, confusion matrix, per-class accuracy và vẽ dự đoán sai. Đây là đánh giá cuối của thực nghiệm, không thuộc Phase 8 cấu hình notebook.



In [ ]:
# -- Đánh giá cả hai model trên test set --
inf_v0 = run_inference(model_v0, test_dl, DEVICE, loss_fn=loss_fn)
result_v0 = evaluate_predictions(inf_v0, CLASS_NAMES)
inf_v1 = run_inference(model_v1, test_dl, DEVICE, loss_fn=loss_fn)
result_v1 = evaluate_predictions(inf_v1, CLASS_NAMES)
inf_ft = run_inference(model_v1_ft, test_dl, DEVICE, loss_fn=loss_fn)
result_ft = evaluate_predictions(inf_ft, CLASS_NAMES)

test_results = pd.DataFrame([
    {'Run': 'v0 — VGG16',          **result_v0},
    {'Run': 'v1 — ResNet-18',       **result_v1},
    {'Run': 'v1 Fine-tuned layer4', **result_ft},
]).set_index('Run')

print('=== TEST SET RESULTS ===')
print(test_results.to_string())

In [ ]:
# -- Confusion matrix cho model tốt nhất --
# (Dùng v1 fine-tuned)
y_pred, y_true = evaluate.predict_dataloader(model_v1_ft, test_dl, device=DEVICE)
cm = evaluate.compute_confusion_matrix(y_true=y_true, y_pred=y_pred, num_classes=NUM_CLASSES)

evaluate.plot_confusion_matrix(
    cm,
    class_names=CLASS_NAMES,
    title='Confusion Matrix — V1 ResNet-18 Fine-tuned (Test Set)',
    show=True,
)

In [ ]:
# -- Per-class accuracy --
class_acc = evaluate.per_class_accuracy(cm)

df_class = pd.DataFrame({
    'Class'   : CLASS_NAMES,
    'Accuracy': [f'{a:.3f}' for a in class_acc.tolist()],
}).sort_values('Accuracy')

print('Per-class accuracy (thấp → cao):')
print(df_class.to_string(index=False))

In [ ]:
# -- Top 5 nhầm lẫn phổ biến nhất --
top5 = evaluate.top_confusions(cm, CLASS_NAMES, k=5)
print('Top 5 nhầm lẫn:')
print(f'  {"True":>12} → {"Predicted":<12}  Count')
print('  ' + '-' * 38)
for true_cls, pred_cls, count in top5:
    print(f'  {true_cls:>12} → {pred_cls:<12}  {count}')

In [ ]:
# -- Visualize một số dự đoán sai --
wrong_mask = (y_pred != y_true)
wrong_indices = wrong_mask.nonzero(as_tuple=True)[0][:9]

# Lấy ảnh gốc từ test_dl (cần rebuild để có ảnh de-normalized)
all_imgs, all_true = [], []
for imgs, lbls in test_dl:
    all_imgs.append(imgs)
    all_true.append(lbls)
all_imgs = torch.cat(all_imgs)

sample_imgs  = [denormalize(all_imgs[i]) for i in wrong_indices]
sample_true  = [y_true[i].item() for i in wrong_indices]
sample_pred  = [y_pred[i].item() for i in wrong_indices]

evaluate.plot_sample_predictions(
    images      = sample_imgs,
    true_labels = sample_true,
    pred_labels = sample_pred,
    class_names = CLASS_NAMES,
    ncols       = 3,
    show        = True,
)

---
## Trạng thái và bước tiếp theo

### Phase 8 — cấu hình notebook

- Chuẩn hóa nguồn code, đường dẫn Drive/runtime, dataset và cấu hình `RUN_ID`.
- Kiểm tra thiết lập cấu hình mà không huấn luyện hay đánh giá test.
- Chưa xác nhận bootstrap/đọc-ghi Drive thật cho đến khi người dùng thực hiện trên Colab.

### Phase thực nghiệm tiếp theo

Sau khi các cell train/resume/reuse được nối với cấu hình sáu run và các đường dẫn artifacts, người dùng chạy từng run theo thứ tự/phụ thuộc trong kế hoạch. Đánh giá test chỉ thực hiện sau khi mỗi run hoàn tất.

